# Compartment calling and evaluation tutorial

This tutorial follows the original [daydayupzzl/compartment source](https://github.com/daydayupzzl/compartment/blob/e068313a9b68ce07357123acfb6c1cbb1d6b3f42/readme.md),
commit `e068313`. The calling scripts, helper modules and original evaluation
notebook are restored from that version.

The workflow reads raw, scHiCluster-imputed and Higashi-imputed contact maps,
calls compartments with CpG, Self-PCA or Ref-PCA, collects their native outputs
and evaluates them with the original notebook. scGHOST follows its separate
upstream workflow at 500 kb.

Commands below are displayed as text. Reading this tutorial does not run a job.
The Lee2019 example uses the original source's dataset key `Lee` and the
cell-type label `L2/3`. Native source files retain their original server paths;
set the dataset's actual resource and output locations before execution.

Use the [shared path setup](tutorials.ipynb#Shared-path-prefix) from the analysis
checkout:

```bash
source tutorials/paths.sh
cd "${BENCHMARK_ROOT}/code"
```


<a id="Inputs-and-evaluation-design"></a>

## 1. Inputs and evaluation design


<a id="1.-Define-the-evaluated-conditions"></a>

### 1.1. Define the native analysis branches

The source readme describes these resolutions and input modalities:

| Resolution | Compartment callers | Input modalities | Levels |
|---|---|---|---|
| 100 kb | CpG, Self-PCA, Ref-PCA | Raw, scHiCluster, Higashi | Single cell, pseudo-bulk |
| 500 kb | The same callers, plus external scGHOST | Raw, scHiCluster, Higashi | Single cell, pseudo-bulk |

Caller and input modality remain separate choices. In the original evaluation,
the method keys are `pca`, `cpg` and `higashi`; the method key `higashi` denotes
Ref-PCA, whereas a datatype named `higashi` denotes the imputed input.

The supplied collection script and result notebook are configured for 100 kb.
Run a separate resolution condition with its matching source paths and resources.


<a id="2.-Prepare-identities-and-exact-sample-membership"></a>

### 1.2. Prepare the original metadata and cell tables

Native calling reads a headerless, two-column table containing the original
cell IDs and their `.cool` paths. The `npy` mode reads the original chromosome
sparse-matrix arrays. Keep the table and sparse-array row order consistent
with the original `label_info.pickle` metadata.

The collection script uses `label_info["cell type"]` to select the target
cell type by row index. Its path conventions include `cell_table_100K.tsv`,
`.compartment.nc` outputs and Ref-PCA HDF5 files.

Configure the actual paths in the native source. The three input modalities
use the same biological sample definitions within each evaluated condition.


<a id="3.-Arrange-the-cell-number-gradient-and-raw-reference"></a>

### 1.3. Arrange the cell-number series and raw reference

The original collection script builds the cell-number series from the selected
type's metadata count: 100, 200, 300, and so on, with the all-cell condition.
Its reference is the selected cell type's all-cell raw pseudo-bulk PCA track:

```text
All raw cells of the selected type → merged raw matrix → Self-PCA track
```

The result notebook reads the reference from
`raw_bulk_num_pca_call[f"{chr_name}_bulkall"]`. Keep that reference key and the
native sample filenames consistent when preparing the cache.

Ref-PCA's merged background used to fit its projection axes is a separate
input to calling; the raw all-cell PCA track above is used for evaluation.


<a id="Compartment-calling"></a>

## 2. Compartment calling


<a id="4.-Call-CpG-based-compartments"></a>

### 2.1. Call CpG-based compartments

The original [CpG caller](https://github.com/daydayupzzl/compartment/blob/e068313a9b68ce07357123acfb6c1cbb1d6b3f42/CpG/calling.py) combines contact weights
with the CpG-density profile for the same genome and resolution. Its core
calculation is:

```python
# matrix is the normalized sparse chromosome matrix.
# cpg_ratio is the matched per-bin CpG-density Series.
comp = matrix.dot(cpg_ratio.values[:, None])[:, 0]
```

Set the genome-resource paths in `main()`, then run the native interface:

```bash
python compartment/tool/CpG/calling.py \
  --dataset Lee --cell_table_path "${COMPARTMENT_CELL_TABLE}" \
  --output_prefix "${COMPARTMENT_CPG_PREFIX}" \
  --resolution 100000 --mode cool --cpu 10
```

The source selects the 100-kb or 500-kb CpG file from `--resolution`.
Samples at the same genome and resolution share that CpG profile.
The native aggregate output is `<output_prefix>.compartment.nc`.


<a id="5.-Call-Self-PCA-compartments"></a>

### 2.2. Call Self-PCA compartments

[Self-PCA](https://github.com/daydayupzzl/compartment/blob/e068313a9b68ce07357123acfb6c1cbb1d6b3f42/Self-PCA/calling.py) fits each target matrix independently.
The native defaults use `sqrtvc` normalization, distance-dependent expectation
correction, a bin-to-bin correlation matrix and two-component PCA. Compartment
contrast selects the principal component; CpG information calibrates its sign.

```bash
python compartment/tool/Self-PCA/calling.py \
  --dataset Lee --cell_table_path "${COMPARTMENT_CELL_TABLE}" \
  --output_prefix "${COMPARTMENT_PCA_PREFIX}" \
  --resolution 100000 --mode cool --cpu 10 \
  --norm_type sqrtvc --exp_type dicrete
```

`dicrete` is the option spelling in the original source. Configure its native
genome and CpG resource paths before calling each raw or imputed sample set.


<a id="6.-Call-Ref-PCA-compartments"></a>

### 2.3. Call Ref-PCA compartments

Ref-PCA retains the original Higashi analysis and backend modules. It builds a
merged raw background, applies the native masks and chromosome-arm split,
fits PCA models and projects the target matrices onto those axes.

Run in the original Higashi environment, with its native JSON and data paths:

```bash
cd "${BENCHMARK_ROOT}/code/compartment/tool/Ref-PCA"

python calling_single_cell.py \
  --config "${COMPARTMENT_CONFIG}" --data_type higashi \
  --calib --output scCompartment.Higashi.100K.hdf5

python calling_bulk.py \
  --config "${COMPARTMENT_CONFIG}" --temp "${COMPARTMENT_TEMP_DIR}" \
  --bulk bulk100 --data_type higashi --calib \
  --output scCompartment_bulk100.hdf5
```

The JSON supplies the dataset, resolution, temporary directory, chromosome list,
neighbor settings and cytoband path. The source also has native paths for
scHiCluster tables and CpG calibration resources; set them for the same run.
Outputs retain the `compartment`, `compartment_raw` and `compartment_zscore` groups.


<a id="7.-Supply-native-scGHOST-outputs-at-500-kb"></a>

### 2.4. Run the separate scGHOST workflow at 500 kb

The source readme links to the [official scGHOST repository](https://github.com/ma-compbio/scGHOST)
for its installation and native single-cell workflow. Follow that workflow for
the 500-kb condition and retain its original cell identities and genomic bins.
The readme describes pseudo-bulk results as bin-level averages of the selected
single-cell outputs.

The restored `get_data.py` and `get_result.ipynb` use the three method keys
`pca`, `cpg` and `higashi`. They do not contain a scGHOST output reader; the
external scGHOST analysis is kept separate from this supplied notebook example.


<a id="8.-Declare-native-tracks-and-contact-sources"></a>

### 2.5. Collect the native caller outputs

After calling, use the original [collection script](https://github.com/daydayupzzl/compartment/blob/e068313a9b68ce07357123acfb6c1cbb1d6b3f42/Evaluation/get_data.py)
to gather the selected type's tracks. It reads the NetCDF and Ref-PCA HDF5 files
using the native metadata and row indices:

```bash
cd "${BENCHMARK_ROOT}/code/compartment/tool/evaluation"
python get_data.py Lee 'L2/3'
```

The cached pickle contains three dictionaries:

| Key | Contents |
|---|---|
| `single_cell_call` | Per-cell tracks by method, input modality and chromosome |
| `bulk_num_call` | Pseudo-bulk tracks across cell-number conditions |
| `raw_bulk_num_pca_call` | Raw pseudo-bulk PCA tracks, including the all-cell reference |

The original output-name rule gives `Lee_L2_3.pkl` for this dataset/type pair.
The script's native path variables determine where that cache is written.


<a id="Structural-and-cell-level-evaluation"></a>

## 3. Structural and cell-level evaluation


<a id="9.-Compare-boundary-locations-and-compartment-tracks"></a>

### 3.1. Compare boundaries and compartment tracks

The original [boundary metric](https://github.com/daydayupzzl/compartment/blob/e068313a9b68ce07357123acfb6c1cbb1d6b3f42/Evaluation/tools.py) optionally denoises
both vectors, then finds their sign-change boundaries. A predicted boundary
counts as matched if any reference boundary lies within one bin. This is the
source's per-prediction matching rule; a reference boundary can be reused.

```python
TP = sum(1 for pb in pred_bound if np.any(np.abs(true_bound - pb) <= tolerance))
```

Precision, recall, F1 and Jaccard follow the original formulas. The notebook also
reports boundary counts. Its track-correlation calculation uses the denoised
predicted vector and the corresponding reference vector:

```python
r, _ = pearsonr(denoised_vec, label_vec)
```

The original notebook summarizes chromosome values within each cell or
pseudo-bulk condition and then reports the corresponding method/input results.


<a id="10.-Calculate-saddle-strength-on-the-matched-contact-map"></a>

### 3.2. Calculate the original saddle strength

The notebook reads the corresponding raw or imputed contact matrix, computes
its original observed/expected representation and calls `compute_saddle`:

```python
oe_mat = get_OE_matrix(contact_mat)
saddle_result = compute_saddle(oe_mat, ab_vec)
```

The source uses `pd.qcut(..., 50, duplicates='drop')` to group the compartment
scores. Saddle contrast uses the first ten and last ten groups, retaining
that fixed rule when repeated scores reduce the available groups.

```text
saddle strength = (sum(AA) + sum(BB)) / (sum(AB) + sum(BA) + 1e-8)
```

The helper also returns the saddle matrix and the four block means `AA`,
`BB`, `AB` and `BA`, using the source's original naming and formulas.


<a id="11.-Evaluate-existing-cell-cluster-assignments"></a>

### 3.3. Use the original cell-clustering utility

The restored [CpG/tools.py](https://github.com/daydayupzzl/compartment/blob/e068313a9b68ce07357123acfb6c1cbb1d6b3f42/CpG/tools.py) contains the original
`clustering_metrics` helper. It fits K-means to the supplied feature matrix,
using `random_state=0`. When `n_clusters` is omitted, it uses the number of
distinct reference cell labels.

```python
sil_score, ch_score, ari, ami, v = clustering_metrics(emb, true_labels)
```

The outputs are silhouette score, Calinski–Harabasz, ARI, AMI and V-measure.
`emb` is the actual cell-by-feature matrix and `true_labels` follows its real
cell order. This utility is separate from the structural-metric cells in
`get_result.ipynb`; the restored source has no `native_labels` configuration mode.


<a id="Run-the-workflow-and-read-outputs"></a>

## 4. Run the workflow and read outputs


<a id="12.-Validate-and-evaluate-the-configured-workflow"></a>

### 4.1. Run the original collection and result notebook

Configure the native paths in the collection script, then collect the completed
calls and open the original [result notebook](https://github.com/daydayupzzl/compartment/blob/e068313a9b68ce07357123acfb6c1cbb1d6b3f42/Evaluation/get_result.ipynb):

```bash
cd "${BENCHMARK_ROOT}/code/compartment/tool/evaluation"
python get_data.py Lee 'L2/3'
jupyter notebook get_result.ipynb
```

In the notebook, set `dataset='Lee'`, `target_cell_type='L2/3'`, the matching
resolution/resources and the populated pickle path. Use its read-data and
metric cells with the cache produced by `get_data.py`.

The source's setup cell also includes a block that writes empty dictionaries
to the cache. Retain the collected pickle rather than rerunning that empty-cache
write block when loading completed caller outputs.


<a id="13.-Read-the-output-files"></a>

### 4.2. Read the native outputs

| Native artifact | Contents |
|---|---|
| CpG/Self-PCA `.compartment.nc` | Cell or sample compartment tracks and genomic bins |
| Ref-PCA `.hdf5` | Quantile, raw and z-score profiles with native bin coordinates |
| Dataset/type `.pkl` cache | The three dictionaries gathered by `get_data.py` |
| Result-notebook tables | Boundary F1/counts, track correlation and saddle summaries |
| `clustering_metrics` return values | Original K-means-based cell-clustering scores |

Output directories and prefixes follow the native script settings. The result
notebook displays its dictionaries and DataFrames; a unified CSV/manifest export
is not part of this restored source.


<a id="14.-Continue-to-the-other-tutorials"></a>

### 4.3. Continue to the other tutorials

[Data preparation](data_preparation.ipynb) describes the input and imputation
workflows. [Embedding](embedding.ipynb), [Domain/TAD](tad.ipynb),
[Loop](loop.ipynb) and [Loop figures](loop_figures.ipynb) cover the other tasks.
Return to the [tutorial index](tutorials.ipynb) to choose a workflow.


<a id="Software-and-implementation"></a>

### 4.4. Software and original implementation

| Stage | Native software context |
|---|---|
| CpG/Self-PCA | NumPy, pandas, SciPy, scikit-learn, Cooler, Xarray, PyTables, joblib, tqdm, tqdm-joblib, UMAP, seaborn and matplotlib |
| CpG resources | Matching genome FASTA, chromosome sizes and bedtools |
| Ref-PCA | Original Higashi environment, PyTorch and the complete bundled modules |
| Structural evaluation | NumPy, pandas, SciPy, h5py, Cooler, Xarray and Jupyter |
| scGHOST | Its official environment |

The original source files are under `compartment/tool/` in the analysis-code
checkout. The source `Evaluation/` directory is named `tool/evaluation/` there;
its Python files and original notebook retain their upstream contents.
See [environment and source notes](https://github.com/rawandy2001-design/benchmark/blob/main/compartment/ENVIRONMENT.md) and the
[pinned source readme](https://github.com/daydayupzzl/compartment/blob/e068313a9b68ce07357123acfb6c1cbb1d6b3f42/readme.md).
